In [1]:
# Exercise 9 : Conversational RAG

from pypdf import PdfReader

reader = PdfReader("os.pdf")
text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"


In [2]:
def chunk_text(text, chunk_size=100, overlap=20):

    words = text.split()
    step = chunk_size - overlap
    chunks = []
    for i in range(0,len(words),step):
        chunk = " ".join(words[i:i+chunk_size])

        if chunk:
            chunks.append(chunk)
    return chunks
    
chunks = chunk_text(text)

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

chunk_embeddings = model.encode(chunks)

W0629 17:38:38.756000 32576 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
C:\Users\uniqu\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
import numpy as np
chunk_embeddings = np.array(
    chunk_embeddings,
    dtype=np.float32
)

In [5]:
import faiss
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(chunk_embeddings)

In [6]:
def retrieve(query, k=3):
    query_embedding = model.encode([query])

    query_embedding = np.array(
        query_embedding,
        dtype=np.float32
    )

    distances, indices = index.search(
        query_embedding,
        k
    )

    return [chunks[i] for i in indices[0]]


In [7]:
def build_prompt(query, retrieved_chunks, chat_history):
    context = "\n".join(retrieved_chunks)
    history = "\n".join(chat_history)
    prompt = f"""
Context :
{context}

History :
{history}

Question :
{query}

Answer using this above context and chat history

"""
    return prompt

In [ ]:
from google import genai

client = genai.Client(api_key="GEMINI_API_KEY")


def rewrite_query(query, recent_questions):
    history = " ".join(recent_questions)
    rewrite_prompt = f"""
Given a conversation history and current question, rewrite the current question
into a standalone question.
History : 
{history}

Current Question : 
{query}

Standalone Question : 
"""
    response = client.models.generate_content(
        model = "gemini-2.5-flash",
        contents = rewrite_prompt
    )

    return response.text.strip()

In [11]:
chat_history = []
recent_questions = []

while True:

    query = input("Ask: ")

    if query == "exit":
        break

    standalone_query = rewrite_query(query, recent_questions)
    retrieved_chunks = retrieve(standalone_query,3)
    
    print("Standalone Query : ")
    print(standalone_query)

    prompt = build_prompt(
        query,
        retrieved_chunks,
        chat_history
    )
    # print(retrieved_chunks)

    response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
    )

    print("Answer : ")
    print(response.text)

    chat_history.append(
        f"User: {query}"
    )

    chat_history.append(
        f"Assistant: {response.text}"
    )

    recent_questions.append(query)
    recent_questions = recent_questions[-3:]
    chat_history = chat_history[-6:]


Ask:  what is paging?


Standalone Query : 
what is paging?
Answer : 
Paging is a memory management technique that creates fixed-size blocks of memory, known as pages. It eliminates external fragmentation but can cause a little internal fragmentation in the last page. In paging, an address is composed of a page number plus an offset, which doesn't necessarily have a direct logical meaning to the programmer.


Ask:  what is demand paging?


Standalone Query : 
what is demand paging?
Answer : 
Demand paging is a technique where a page is brought into memory only when it is first accessed, rather than loading all pages upfront. If a process attempts to access a page that is not currently in memory, the hardware raises a page fault, and the operating system then finds that page on disk and loads it into memory.


Ask:  what is diff between them?


Standalone Query : 
What is the difference between paging and demand paging?
Answer : 
Paging is a memory management technique that divides a process's address space and physical memory into fixed-size blocks called pages. It's about how memory is *structured* and addressed (page number + offset).

Demand paging, on the other hand, is a *strategy* used within a paging system (often in conjunction with virtual memory). It dictates *when* pages are loaded into physical memory: a page is brought in only when it is first accessed, rather than loading all pages upfront. If an accessed page isn't in memory, it triggers a page fault, and the OS then loads it from disk.

In short, paging is the underlying method of organizing memory into pages, while demand paging is the method of *loading* those pages into memory only when they are actually needed.


Ask:  exit
